## Sample project with CrewAI
- to create a sample resume checking application
- Read the pdf resume file and check it with a JD
- find the revelvence of the resume agains the job description
- re-write the resmue based on the job description to pass the ATS

In [4]:
%pip install crewai | tail -n 1
%pip install crewai-tools | tail -n 1
%pip install litellm | tail -n 1
%pip install docling| tail -n 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires chromadb<2.0.0,>=1.3.5, but you have chromadb 1.1.1 which is incompatible.
litellm 1.83.10 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
litellm 1.83.10 requires python-dotenv==1.0.1, but you have python-dotenv 1.1.1 which is incompatible.
crewai-tools 1.14.2 requires tiktoken~=0.8.0, but you have tiktoken 0.12.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.10 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
litellm 1.83.10 requires python-dotenv==1.0.1, but you have python-dotenv 1.1.1 whic

## Tools
- a duck duck go search tool

In [1]:
from ddgs import DDGS
from crewai.tools import tool

@tool
def search_tool(query, max_results=3):
    """tool to search web"""
    with DDGS() as search:
        results = [r for r in 
        search.text(
            query, 
            max_results=max_results,
            safe_search=True
        )]
    
    return results

# to test the tool
search_tool.run("who was the fist president of USA?")


[{'title': 'George Washington - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/George_Washington',
  'body': '4 days ago -George Washington(February 22, 1732 [O.S. February 11, 1731] – December 14, 1799) was a Founding Father and the first president of the United States, serving from 1789 to 1797. As commander of the Continental Army, Washington led Patriot forces to victory in the American Revolutionary War ...'},
 {'title': 'First inauguration of George Washington - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/First_inauguration_of_George_Washington',
  'body': 'February 19, 2026 -The first inauguration ofGeorge Washingtonas the first president of the United States was held on Thursday, April 30, 1789, on the balcony of Federal Hall in New York City. The inauguration was held nearly two months after the beginning of ...'},
 {'title': 'Chronological List of Presidents, First Spouses, and Vice Presidents of the United States - Presidents of the United States: Selected Ima

## LLM
- need to use a Groq model in Crewai

In [ ]:
import os
from crewai import LLM
import dotenv

dotenv.load_dotenv("../.env")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ")

llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0
)

#llm call should be a list of dictionaries
# dictionary content = role and message
llm.call([{
    "role": "user",
    "content": "which is the capital of Hessen, Germany"
}])

'The capital of Hessen, Germany is Wiesbaden.'

In [5]:
# Resume and JD

# need to input a resume (pdf)
# convert the resume to markdown
import os
from docling.document_converter import DocumentConverter

resume_path = "../data/resume.pdf"

def convert_to_markdown(doc_path):
    if not os.path.exists(doc_path):
        raise FileNotFoundError(f"Document not found at path: {doc_path}")
    converter = DocumentConverter()
    markdown = converter.convert(doc_path).document.export_to_markdown()
    return markdown

markdown_resume = convert_to_markdown(resume_path)
# print(markdown_resume)

# JD
jd_path = "../data/jd.txt"

def extract_jd(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"JD file not found at path: {path}")
    with open(path, "r") as f:
        jd = f.read()

    return jd

jd = extract_jd(jd_path)
# print(jd)

In [ ]:
from crewai import Agent, Task

carrier_coach = Agent(
    role = "Senior_Carrier_Coach",
    goal = """
    Your goal is to analyze the resume and find the key strengths and weakness in the resme
    - Objectively analyze the resume and find the strengths
    - Find the gaps in the resume, and potential improvement avenues
    """,
    back_story = """
    You are a senior hr person, have extensive experience in Tech-recurting
    you have recruted many candidates for top Technology firms like Meta, Google, and Apple
    """,
    llm = llm,
    verbose=True,
    delegation=False
    tools=[search_tool]
)

resume_check = Task(
    descripton="""
    
    """,
    expected_output=,
    agent=carrier_coach
)

